# Masked-Diffusion BabyLM — Strict-Small (Training)

Train a LLaDA/MDLM-style masked-diffusion language model on the BabyLM 2026
**Strict-Small** corpus (≤10M words, ≤10 epochs, ≤100M words seen).

Everything heavy (data, tokenizer, runs) lives on Google Drive so a Colab
disconnect never loses work. Data prep is skipped if already done, and training
**auto-resumes** from the last checkpoint — re-run the training cell after any
disconnect (even on another day) and it continues where it left off. Run the
cells top to bottom.

**Before you start:** add an `HF_TOKEN` (read access) via the Colab Secrets panel (key icon).

In [ ]:
# Step 1 — Parameters (edit these)
REPO_URL   = "https://github.com/Amos-Luna/Masked-Diffusion-BabyLM.git"
DRIVE_ROOT = "/content/drive/MyDrive/Researchs/BabyLM_diffusion_G4"
CONDITION  = "MD_base"   # MD_base | MD_freq_mask | MD_layerdup
SEED       = 42
# True = new run with current pipeline (--no-resume). False = continue latest run.
FRESH_RUN  = True

In [ ]:
# Step 2 — Mount Drive + clone/pull the repo (always pull for latest training fixes)
from google.colab import drive
drive.mount("/content/drive")

import os, subprocess
if not os.path.isdir("/content/Masked-Diffusion-BabyLM"):
    subprocess.run(["git", "clone", REPO_URL, "/content/Masked-Diffusion-BabyLM"], check=True)
%cd /content/Masked-Diffusion-BabyLM/diffusion
subprocess.run(["git", "pull"], check=True)
print("Repo updated. Training uses: block shuffle, stride dev, LLaDA loss, manifest budget.")

In [ ]:
# Step 3 — HuggingFace token (from Colab Secrets)
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded.")
except Exception as e:
    print("Set HF_TOKEN in Colab Secrets (key icon).", e)

In [ ]:
# Step 4 — Bootstrap: install + Drive symlinks + data + smoke test
!bash scripts/colab_bootstrap.sh --drive-root "$DRIVE_ROOT"

# Show corpus stats used for the CFP word budget (train.py reads this manifest).
import json, os
mf = "data/tokens/manifest.json"
if os.path.isfile(mf):
    m = json.load(open(mf))
    ratio = m["n_words"] / m["n_tokens"]
    print(f"\nCorpus: {m['n_words']:,} words / {m['n_tokens']:,} tokens "
          f"(words_per_token={ratio:.4f})")
    print(f"CFP cap: min(100M words seen, 10 epochs) = min(100M, {10*m['n_words']:,})")

In [ ]:
# Step 5 — Train. Checkpoints land in runs/<YYYY-MM-DD>_{condition}_seed{S}/ on Drive.
# FRESH_RUN=True adds --no-resume (required after pipeline fixes). Otherwise
# re-run this cell after a disconnect to continue the same run.
# NOTE: must be a `!` shell command (not subprocess.run) so the step/loss/CE/LR
# logs STREAM live into this cell — subprocess.run writes to the kernel's stdout,
# which Colab does not display.
RESUME_FLAG = "--no-resume" if FRESH_RUN else ""
print("FRESH run (--no-resume)." if FRESH_RUN else "Resuming latest checkpoint, if any.")
!python scripts/train.py --condition $CONDITION --seed $SEED -v \
    --token-data data/tokens --tokenizer tokenizer/mdlm_bpe_16k $RESUME_FLAG

In [ ]:
# Step 6 — Training curves from the run log.
# Watch UNWEIGHTED val CE for real progress (low variance). Weighted MDLM loss
# magnitude changed with LLaDA normalization — only compare weighted loss within
# the same run, not across old vs new pipeline runs. LR schedule: warmup + cosine.
# PNG saved into the run dir on Drive for the paper.
import glob, json, os
import matplotlib.pyplot as plt

# Pick the latest REAL run (skip the bootstrap smoke run, which also matches).
runs = [d for d in sorted(glob.glob(f"runs/*_{CONDITION}_seed{SEED}"))
        if not os.path.basename(d).startswith("_smoke")]
run = runs[-1]
steps, train, ce, lr = [], [], [], []
eval_steps, val, val_ce = [], [], []
for line in open(f"{run}/log.jsonl"):
    r = json.loads(line)
    if r["phase"] == "train":
        steps.append(r["step"]); train.append(r["avg_train_loss"])
        ce.append(r.get("avg_ce_unweighted")); lr.append(r.get("lr"))
    else:
        eval_steps.append(r["step"]); val.append(r["val_loss"])
        val_ce.append(r.get("val_ce_unweighted"))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.plot(steps, train, alpha=0.35, label="train loss (weighted 1/t)")
if any(c is not None for c in ce):
    ax1.plot(steps, ce, label="train CE (unweighted)")
if any(c is not None for c in val_ce):
    ax1.plot(eval_steps, val_ce, "o-", label="val CE (unweighted)")
else:
    ax1.plot(eval_steps, val, "o-", label="val loss (weighted)")
ax1.set_xlabel("step"); ax1.set_ylabel("loss / CE"); ax1.legend(); ax1.set_title(os.path.basename(run))
if any(v is not None for v in lr):
    ax2.plot(steps, lr); ax2.set_xlabel("step"); ax2.set_ylabel("learning rate")
    ax2.set_title("LR schedule (warmup + cosine)")

os.makedirs(f"{run}/figures", exist_ok=True)
out = f"{run}/figures/train_curve.png"
plt.savefig(out, dpi=150, bbox_inches="tight")
print("Saved", out)   # lives on Drive — survives disconnects
plt.show()